# Evaluation of audio-visual classifiers

Code to generate the confusion matrix was adapted from this tutorial: https://medium.com/@gubrani.sanya2/evaluating-multi-class-classification-model-using-confusion-matrix-in-python-4d9344084dfa <br>
Some of the parameters for the confusion matrix were adapted from: https://medium.com/@dtuk81/confusion-matrix-visualization-fc31e3f30fea

In [ ]:
from project_config import DATA_DIR
import os
import re
from sklearn.metrics import classification_report, confusion_matrix
from ultimate_wav2vec import AudioClassifier
from ultimate_image_classification import classify_image

# audio input: remove "_input_1.wav"
# audio embedding: remove "_1.wav"
# image input: "_input_1.jpeg"
# image embedding: "_1.jpeg"
# remove all _.

# ChatGPT generated the def get_true_label() as I couldn't get the correct string replacements despite multiple tries unfortunately
def get_true_label(file_name):
    cleaned_name = re.sub(r'_input_\d+', '', file_name)  # Remove patterns like '_input_1'
    cleaned_name = re.sub(r'_\d+', '', cleaned_name)     # Remove patterns like '_1'
    cleaned_name = re.sub(r'_', ' ', cleaned_name)       # Replace all underscores with spaces
    cleaned_name = re.sub(r'\.\w+$', '', cleaned_name)   # Remove file extension
    return cleaned_name.strip()  # Remove any leading/trailing whitespace

image_predictions = []
image_true_labels = []
audio_predictions = []
audio_true_labels = []

'''
print(image_predictions)
print (audio_predictions)
'''

# this was partially copied from the other script, ultimate_connect, only without the ngrok url part
def process_files(examples_folder):
    audio_classifier = AudioClassifier()
    for category_folder in os.listdir(examples_folder):
        category_path = os.path.join(examples_folder, category_folder)
        if os.path.isdir(category_path):
            for file in os.listdir(category_path):
                file_path = os.path.join(category_path, file)
                true_label = get_true_label(file)
                if file.endswith(".wav"):
                    reference_embeddings = audio_classifier.create_embeddings_for_folder(str(DATA_DIR / 'audios_categories'))
                    similarities = audio_classifier.classify_audio(file_path, reference_embeddings)
                    if similarities:
                        best_match = max(similarities, key=similarities.get)
                    else:
                        best_match = "No Audio" # now dealt with by qwen in the same way as the nothing category
                    best_match_cleaned = get_true_label(best_match)  # clean predicted label
                    similarity_score = similarities.get(best_match, 0)
                    audio_predictions.append(best_match_cleaned)
                    audio_true_labels.append(true_label)
                    print(f"audio file: {true_label}, predicted: {best_match_cleaned}, ismilarity: {similarity_score:.3f}")
                elif file.endswith(".jpeg"):
                    predicted_label, similarity_score = classify_image(file_path)
                    predicted_label_cleaned = get_true_label(predicted_label)
                    image_predictions.append(predicted_label_cleaned)
                    image_true_labels.append(true_label)
                    print(f"image file: {true_label}, predicted: {predicted_label_cleaned}, similarity: {similarity_score:.3f}")

    print("confusion matrix for audio:")
    print(confusion_matrix(audio_true_labels, audio_predictions))
    print("classification report for audio:")
    print(classification_report(audio_true_labels, audio_predictions))
    print("confusion matrix for images:")
    print(confusion_matrix(image_true_labels, image_predictions))
    print("classification report for images:")
    print(classification_report(image_true_labels, image_predictions))
process_files(str(DATA_DIR / 'examples'))

In [ ]:
print (audio_true_labels)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def split_labels(labels):
    return ['\n'.join(label.split()) for label in labels] # to fix the category names because some of them are excessively long

def plot_confusion_matrices(audio_true_labels, audio_predictions, image_true_labels, image_predictions):
    audio_cm = confusion_matrix(audio_true_labels, audio_predictions)
    image_cm = confusion_matrix(image_true_labels, image_predictions) # confusion matrices
    unique_audio_labels = sorted(set(audio_true_labels + audio_predictions)) # labels are split
    unique_image_labels = sorted(set(image_true_labels + image_predictions))
    split_audio_labels = split_labels(unique_audio_labels)
    split_image_labels = split_labels(unique_image_labels)

    print (split_audio_labels)
    print (split_image_labels)

    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    sns.heatmap(audio_cm, annot=True, fmt="d", cmap="Blues", ax=ax[0], xticklabels=split_audio_labels, yticklabels=split_audio_labels)
    ax[0].set_title("confusion matrix for audio")
    ax[0].set_xlabel("predicted labels")
    ax[0].set_ylabel("true labels")
    sns.heatmap(image_cm, annot=True, fmt="d", cmap="Blues", ax=ax[1], xticklabels=split_image_labels, yticklabels=split_image_labels)
    ax[1].set_title("confusion matrix for images")
    ax[1].set_xlabel("predicted labels")
    ax[1].set_ylabel("true labels")
    plt.tight_layout()
    plt.show()
plot_confusion_matrices(audio_true_labels, audio_predictions, image_true_labels, image_predictions)